In [0]:
# ============================================================

# GOLD – dim_sla

# Grain: (source_user_id, project, priority_code)

# ============================================================

from pyspark.sql import functions as F

from pyspark.sql.window import Window

# ---------------- CONFIG ----------------

SILVER_SLA_TABLE = "workspace.slainte_silver.sla_silver_demo"

GOLD_DB = "workspace.slainte_gold"

DIM_SLA = f"{GOLD_DB}.dim_sla"

# ---------------- SETUP ----------------

spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

df_sla = spark.table(SILVER_SLA_TABLE)

# ============================================================

# 1️⃣ Normalize SLA input

# ============================================================

df_sla_base = (

    df_sla

    .select(

        F.col("source_user_id"),

        F.col("project"),

        F.col("priority_level").cast("int").alias("priority_code"),

        F.col("response_time_minutes").cast("int"),

        F.col("resolution_time_hours").cast("int")

    )

    .filter(

        F.col("source_user_id").isNotNull() &

        F.col("project").isNotNull() &

        F.col("priority_code").isNotNull()

    )

    .distinct()

)

# ============================================================

# 2️⃣ Generate SLA surrogate key (partitioned correctly)

# ============================================================

sla_window = (

    Window

    .partitionBy("source_user_id", "project")

    .orderBy("priority_code")

)

df_dim_sla = (

    df_sla_base

    .withColumn("sla_id", F.row_number().over(sla_window))

    .withColumn("created_at", F.current_timestamp())

    .select(

        "sla_id",

        "source_user_id",

        "project",

        "priority_code",

        "response_time_minutes",

        "resolution_time_hours"

    )

)

# ============================================================

# 3️⃣ Write GOLD dimension

# ============================================================

df_dim_sla.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(DIM_SLA)

# ============================================================

# 4️⃣ Preview

# ============================================================

print("✅ dim_sla created successfully")

display(

    spark.table(DIM_SLA)

         .orderBy("source_user_id", "project", "priority_code")

)
 